In [3]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

import operator
from functools import reduce
from functools import partial

torch.manual_seed(0)
np.random.seed(0)
import torch
from torch import nn

def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params
################################################################
# Fourier Layer
################################################################

class SpectralConv2d_fast(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super(SpectralConv2d_fast, self).__init__()

        """
        2D Fourier layer. It performs FFT, linear transform, and Inverse FFT.
        """

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1  # Number of Fourier modes to multiply, at most floor(N/2) + 1
        self.modes2 = modes2

        self.scale = (1 / (in_channels * out_channels))
        # Initialize weights with complex numbers
        self.weights1 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes1, self.modes2, dtype=torch.cfloat))

    # Complex multiplication
    def compl_mul2d(self, input, weights):
        # (batch, in_channel, x, y), (in_channel, out_channel, x, y) -> (batch, out_channel, x, y)
        return torch.einsum("bixy,ioxy->boxy", input, weights)

    def forward(self, x):
        batchsize = x.shape[0]
        # Compute Fourier coefficients up to factor of e^(- something)
        x_ft = torch.fft.rfft2(x, norm='forward')

        # Multiply relevant Fourier modes
        out_ft = torch.zeros(batchsize, self.out_channels,  x.size(-2), x.size(-1)//2 + 1, dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes1, :self.modes2] = \
            self.compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = \
            self.compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)

        # Return to physical space
        x = torch.fft.irfft2(out_ft, s=(x.size(-2), x.size(-1)), norm='forward')
        return x

################################################################
# Fourier Neural Operator 2D
################################################################

class FNO2d(nn.Module):
    def __init__(self, modes1, modes2, width, C_in=1, C_out=1):
        super(FNO2d, self).__init__()

        """
        FNO2d model adapted to inputs with variable channels and spatial resolution.

        Expected input shape: (batch_size, time_steps=1, channels=C_in, height=H, width=W)
        Output shape: (batch_size, time_steps=1, channels=C_out, height=H, width=W)
        """

        self.modes1 = modes1  # Number of Fourier modes in x direction
        self.modes2 = modes2  # Number of Fourier modes in y direction
        self.width = width
        self.padding = 2  # Padding for non-periodic input, can be adjusted

        self.C_in = C_in
        self.C_out = C_out

        self.fc0 = nn.Linear(self.C_in + 2, self.width)  # Input channels + 2 coordinates (x, y)

        self.conv0 = SpectralConv2d_fast(self.width, self.width, self.modes1, self.modes2)
        self.conv1 = SpectralConv2d_fast(self.width, self.width, self.modes1, self.modes2)
        self.conv2 = SpectralConv2d_fast(self.width, self.width, self.modes1, self.modes2)
        self.conv3 = SpectralConv2d_fast(self.width, self.width, self.modes1, self.modes2)

        self.w0 = nn.Conv2d(self.width, self.width, kernel_size=1)
        self.w1 = nn.Conv2d(self.width, self.width, kernel_size=1)
        self.w2 = nn.Conv2d(self.width, self.width, kernel_size=1)
        self.w3 = nn.Conv2d(self.width, self.width, kernel_size=1)

        self.fc1 = nn.Linear(self.width, 128)
        self.fc2 = nn.Linear(128, self.C_out)  # Output channels

    def forward(self, x):
        # Expected input shape: (B, T, C_in, H, W)
        B, T, C_in, H, W = x.shape
        # Since T=1, we can squeeze the time dimension
        x = x.squeeze(1)  # Remove the time dimension, shape becomes (B, C_in, H, W)
        x = x.permute(0, 2, 3, 1)  # Permute to shape (B, H, W, C_in)

        # Get grid and concatenate with input
        grid = self.get_grid(x.shape, x.device)  # Shape: (B, H, W, 2)
        x = torch.cat((x, grid), dim=-1)  # Shape becomes (B, H, W, C_in+2)

        x = self.fc0(x)
        x = x.permute(0, 3, 1, 2)  # Shape: (B, width, H, W)

        x1 = self.conv0(x)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv1(x)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv2(x)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv3(x)
        x2 = self.w3(x)
        x = x1 + x2

        x = x.permute(0, 2, 3, 1)  # Shape: (B, H, W, width)
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)  # Shape: (B, H, W, C_out)

        # Adjust output shape to (B, T, C_out, H, W)
        x = x.permute(0, 3, 1, 2)  # Shape: (B, C_out, H, W)
        x = x.unsqueeze(1)  # Add time dimension T=1, shape becomes (B, 1, C_out, H, W)
        return x  # Final output shape: (B, T, C_out, H, W)

    def get_grid(self, shape, device):
        '''
        Returns a grid of shape (batchsize, H, W, 2)
        '''
        batchsize, size_x, size_y, _ = shape
        gridx = torch.linspace(0, 1, steps=size_x, device=device)
        gridx = gridx.view(1, size_x, 1, 1).repeat(batchsize, 1, size_y, 1)
        gridy = torch.linspace(0, 1, steps=size_y, device=device)
        gridy = gridy.view(1, 1, size_y, 1).repeat(batchsize, size_x, 1, 1)
        return torch.cat((gridx, gridy), dim=-1)  # Shape: (B, H, W, 2)

################################################################
# Testing the Model with New Dimensions
################################################################

if __name__ == '__main__':

    modes1 = 16   # Adjusted modes, should be less than or equal to H/2
    modes2 = 16   # Adjusted modes, should be less than or equal to W/2
    width = 64    # Width of the neural network
    batch_size = 1  # You can adjust the batch size as needed

    T = 1         # Time steps, remains 1 as per your data
    C_in = 1      # Input channels, adjusted to 1
    C_out = 1     # Output channels, adjusted to 1
    H, W = 128, 128  # Height and Width of the input, adjusted to 128 x 128

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = FNO2d(modes1, modes2, width, C_in=C_in, C_out=C_out).to(device)

    x = torch.randn(batch_size, T, C_in, H, W).to(device)
    print("Input shape:", x.shape)
    # Forward pass
    output = model(x)
    print("Output shape:", output.shape)

    print(f"Output shape: {output.shape}")
    total_params = count_parameters(model)
    print(f"Total parameters: {total_params:,}")
    print(f"Parameters in M: {total_params / 1e6:.2f}M")
    print(f"Parameters in B: {total_params / 1e9:.2f}B")

Input shape: torch.Size([1, 1, 1, 128, 128])
Output shape: torch.Size([1, 1, 1, 128, 128])
Output shape: torch.Size([1, 1, 1, 128, 128])
Total parameters: 8,413,953
Parameters in M: 8.41M
Parameters in B: 0.01B


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils import data

def conv(in_planes, output_channels, kernel_size, stride, dropout_rate):
    return nn.Sequential(
        nn.Conv2d(in_planes, output_channels, kernel_size=kernel_size,
                  stride=stride, padding=(kernel_size - 1) // 2, bias=False),
        nn.BatchNorm2d(output_channels),
        nn.LeakyReLU(0.1, inplace=True),
        nn.Dropout(dropout_rate)
    )

def deconv(input_channels, output_channels):
    return nn.Sequential(
        nn.ConvTranspose2d(input_channels, output_channels, kernel_size=4,
                           stride=2, padding=1),
        nn.LeakyReLU(0.1, inplace=True)
    )

def output_layer(input_channels, output_channels, kernel_size, stride, dropout_rate):
    return nn.Conv2d(input_channels, output_channels, kernel_size=kernel_size,
                     stride=stride, padding=(kernel_size - 1) // 2)

class U_net(nn.Module):
    def __init__(self, input_channels, output_channels, kernel_size, dropout_rate):
        super(U_net, self).__init__()
        self.input_channels = input_channels
        self.conv1 = conv(input_channels, 64, kernel_size=kernel_size, stride=2, dropout_rate=dropout_rate)
        self.conv2 = conv(64, 128, kernel_size=kernel_size, stride=2, dropout_rate=dropout_rate)
        self.conv3 = conv(128, 256, kernel_size=kernel_size, stride=2, dropout_rate=dropout_rate)
        self.conv3_1 = conv(256, 256, kernel_size=kernel_size, stride=1, dropout_rate=dropout_rate)
        self.conv4 = conv(256, 512, kernel_size=kernel_size, stride=2, dropout_rate=dropout_rate)
        self.conv4_1 = conv(512, 512, kernel_size=kernel_size, stride=1, dropout_rate=dropout_rate)
        self.conv5 = conv(512, 1024, kernel_size=kernel_size, stride=2, dropout_rate=dropout_rate)
        self.conv5_1 = conv(1024, 1024, kernel_size=kernel_size, stride=1, dropout_rate=dropout_rate)

        self.deconv4 = deconv(1024, 256)
        self.deconv3 = deconv(768, 128)
        self.deconv2 = deconv(384, 64)
        self.deconv1 = deconv(192, 32)
        self.deconv0 = deconv(96, 16)

        self.output_layer = output_layer(16 + input_channels, output_channels,
                                         kernel_size=kernel_size, stride=1, dropout_rate=dropout_rate)

    def forward(self, x):
        # x is expected to be of shape (B, T, C, H, W)
        B, T, C, H, W = x.size()
        assert T == 1, "Expected T=1 since TCHW=1 2 64 448"
        x = x.squeeze(1)  # Remove the T dimension; x is now of shape (B, C, H, W)

        # Encoder path (downsampling)
        out_conv1 = self.conv1(x)     # (B, 64, H/2, W/2)
        out_conv2 = self.conv2(out_conv1)  # (B, 128, H/4, W/4)
        out_conv3 = self.conv3_1(self.conv3(out_conv2))  # (B, 256, H/8, W/8)
        out_conv4 = self.conv4_1(self.conv4(out_conv3))  # (B, 512, H/16, W/16)
        out_conv5 = self.conv5_1(self.conv5(out_conv4))  # (B, 1024, H/32, W/32)

        # Decoder path (upsampling)
        out_deconv4 = self.deconv4(out_conv5)  # (B, 256, H/16, W/16)
        concat4 = torch.cat((out_conv4, out_deconv4), 1)  # (B, 768, H/16, W/16)
        out_deconv3 = self.deconv3(concat4)  # (B, 128, H/8, W/8)
        concat3 = torch.cat((out_conv3, out_deconv3), 1)  # (B, 384, H/8, W/8)
        out_deconv2 = self.deconv2(concat3)  # (B, 64, H/4, W/4)
        concat2 = torch.cat((out_conv2, out_deconv2), 1)  # (B, 192, H/4, W/4)
        out_deconv1 = self.deconv1(concat2)  # (B, 32, H/2, W/2)
        concat1 = torch.cat((out_conv1, out_deconv1), 1)  # (B, 96, H/2, W/2)
        out_deconv0 = self.deconv0(concat1)  # (B, 16, H, W)
        concat0 = torch.cat((x, out_deconv0), 1)  # (B, 16 + input_channels, H, W)

        out = self.output_layer(concat0)  # (B, output_channels, H, W)
        out = out.unsqueeze(1)  # Add the T dimension back; now out is of shape (B, T, output_channels, H, W)
        return out

input_channels = 1     # As per your input dimension C=2
output_channels = 1    # Assuming you want the output to have the same number of channels
kernel_size = 3        # You can adjust this value as needed
dropout_rate = 0.5     # You can adjust this value as needed

model = U_net(input_channels, output_channels, kernel_size, dropout_rate)

# Example input tensor with dimensions BTCHW = (Batch, Time=1, Channels=2, Height=64, Width=448)
B = 4   # Example batch size
T = 1
C = 1
H = 128
W = 128
input_tensor = torch.randn(B, T, C, H, W)

# Pass the input tensor through the model
output = model(input_tensor)

print(f"Input shape: {input_tensor.shape}")   # Should print torch.Size([B, 1, 2, 64, 448])
print(f"Output shape: {output.shape}")        # Should print torch.Size([B, 1, 2, 64, 448])
total_params = count_parameters(model)
print(f"Total parameters: {total_params:,}")
print(f"Parameters in M: {total_params / 1e6:.2f}M")
print(f"Parameters in B: {total_params / 1e9:.2f}B")

Input shape: torch.Size([4, 1, 1, 128, 128])
Output shape: torch.Size([4, 1, 1, 128, 128])
Total parameters: 24,945,226
Parameters in M: 24.95M
Parameters in B: 0.02B


In [6]:
import torch
from torch import nn
import math
from timm.layers import DropPath, trunc_normal_

def stride_generator(N, reverse=False):
    strides = [1, 2] * 10
    if reverse:
        return list(reversed(strides[:N]))
    else:
        return strides[:N]
    
class MLP(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super(MLP, self).__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

class ConvMLP(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super(ConvMLP, self).__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Conv2d(in_features, hidden_features, 1)
        self.act = act_layer()
        self.fc2 = nn.Conv2d(hidden_features, out_features, 1)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, qk_scale=None, attn_drop=0., proj_drop=0.):
        super(Attention, self).__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = (
            self.qkv(x)
            .reshape(B, N, 3, self.num_heads, C // self.num_heads)
            .permute(2, 0, 3, 1, 4)
        )
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class ConvBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads=4,
        mlp_ratio=4.,
        qkv_bias=False,
        qk_scale=None,
        drop=0.,
        attn_drop=0.,
        drop_path=0.,
        act_layer=nn.GELU,
        norm_layer=nn.LayerNorm
    ):
        super(ConvBlock, self).__init__()
        self.pos_embed = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)
        self.norm1 = nn.BatchNorm2d(dim)
        self.conv1 = nn.Conv2d(dim, dim, 1)
        self.conv2 = nn.Conv2d(dim, dim, 1)
        self.attn = nn.Conv2d(dim, dim, 5, padding=2, groups=dim)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = nn.BatchNorm2d(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = ConvMLP(
            in_features=dim,
            hidden_features=mlp_hidden_dim,
            act_layer=act_layer,
            drop=drop
        )

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.LayerNorm, nn.GroupNorm, nn.BatchNorm2d)):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = (
                m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            )
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    @torch.jit.ignore
    def no_weight_decay(self):
        return {}

    def forward(self, x):
        x = x + self.pos_embed(x)
        x = x + self.drop_path(
            self.conv2(self.attn(self.conv1(self.norm1(x))))
        )
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

class SelfAttentionBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        mlp_ratio=4.,
        qkv_bias=False,
        qk_scale=None,
        drop=0.,
        attn_drop=0.,
        drop_path=0.,
        init_value=1e-6,
        act_layer=nn.GELU,
        norm_layer=nn.LayerNorm
    ):
        super(SelfAttentionBlock, self).__init__()
        self.pos_embed = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)
        self.norm1 = norm_layer(dim)
        self.attn = Attention(
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            qk_scale=qk_scale,
            attn_drop=attn_drop,
            proj_drop=drop
        )
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = MLP(
            in_features=dim,
            hidden_features=mlp_hidden_dim,
            act_layer=act_layer,
            drop=drop
        )
        self.gamma_1 = nn.Parameter(init_value * torch.ones((dim)), requires_grad=True)
        self.gamma_2 = nn.Parameter(init_value * torch.ones((dim)), requires_grad=True)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, (nn.LayerNorm, nn.GroupNorm, nn.BatchNorm2d)):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    @torch.jit.ignore
    def no_weight_decay(self):
        return {'gamma_1', 'gamma_2'}

    def forward(self, x):
        x = x + self.pos_embed(x)
        B, N, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = x + self.drop_path(self.gamma_1 * self.attn(self.norm1(x)))
        x = x + self.drop_path(self.gamma_2 * self.mlp(self.norm2(x)))
        x = x.transpose(1, 2).reshape(B, N, H, W)
        return x

def UniformerSubBlock(
    embed_dims,
    mlp_ratio=4.,
    drop=0.,
    drop_path=0.,
    init_value=1e-6,
    block_type='Conv'
):
    assert block_type in ['Conv', 'MHSA']
    if block_type == 'Conv':
        # return ConvBlock(dim=embed_dims, mlp_ratio=mlp_ratio, drop=drop, drop_path=drop_path)
        return SelfAttentionBlock(
            dim=embed_dims,
            num_heads=8,
            mlp_ratio=mlp_ratio,
            qkv_bias=True,
            drop=drop,
            drop_path=drop_path,
            init_value=init_value
        )
    else:
        return SelfAttentionBlock(
            dim=embed_dims,
            num_heads=8,
            mlp_ratio=mlp_ratio,
            qkv_bias=True,
            drop=drop,
            drop_path=drop_path,
            init_value=init_value
        )

class SpatioTemporalEvolutionBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        input_resolution=None,
        mlp_ratio=8.,
        drop=0.0,
        drop_path=0.0,
        layer_i=0
    ):
        super(SpatioTemporalEvolutionBlock, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        block_type = 'MHSA' if in_channels == out_channels and layer_i > 0 else 'Conv'
        self.block = UniformerSubBlock(
            in_channels,
            mlp_ratio=mlp_ratio,
            drop=drop,
            drop_path=drop_path,
            block_type=block_type
        )

        if in_channels != out_channels:
            self.reduction = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=1,
                padding=0
            )

    def forward(self, x):
        z = self.block(x)
        if self.in_channels != self.out_channels:
            z = self.reduction(z)
        return z

class Latent_Dynamical_Core(nn.Module):
    def __init__(
        self,
        channel_in,
        channel_hid,
        N2,
        input_resolution=None,
        mlp_ratio=4.,
        drop=0.0,
        drop_path=0.1
    ):
        super(Latent_Dynamical_Core, self).__init__()
        assert N2 >= 2 and mlp_ratio > 1
        self.N2 = N2
        dpr = [x.item() for x in torch.linspace(1e-2, drop_path, self.N2)]

        evolution_layers = [SpatioTemporalEvolutionBlock(
            channel_in,
            channel_hid,
            input_resolution,
            mlp_ratio=mlp_ratio,
            drop=drop,
            drop_path=dpr[0],
            layer_i=0
        )]

        for i in range(1, N2 - 1):
            evolution_layers.append(SpatioTemporalEvolutionBlock(
                channel_hid,
                channel_hid,
                input_resolution,
                mlp_ratio=mlp_ratio,
                drop=drop,
                drop_path=dpr[i],
                layer_i=i
            ))

        evolution_layers.append(SpatioTemporalEvolutionBlock(
            channel_hid,
            channel_in,
            input_resolution,
            mlp_ratio=mlp_ratio,
            drop=drop,
            drop_path=drop_path,
            layer_i=N2 - 1
        ))
        self.enc = nn.Sequential(*evolution_layers)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.reshape(B, T * C, H, W)
        z = x
        for i in range(self.N2):
            z = self.enc[i](z)
        y = z.reshape(B, T, C, H, W)
        return y

class BasicConv2d(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride,
        padding,
        transpose=False,
        act_norm=False
    ):
        super(BasicConv2d, self).__init__()
        self.act_norm = act_norm
        if not transpose:
            self.conv = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding
            )
        else:
            self.conv = nn.ConvTranspose2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                output_padding=stride // 2
            )
        self.norm = nn.GroupNorm(2, out_channels)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        y = self.conv(x)
        if self.act_norm:
            y = self.act(self.norm(y))
        return y

class ConvDynamicsLayer(nn.Module):
    def __init__(self, C_in, C_out, stride, transpose=False, act_norm=True):
        super(ConvDynamicsLayer, self).__init__()
        if stride == 1:
            transpose = False
        self.conv = BasicConv2d(
            C_in,
            C_out,
            kernel_size=3,
            stride=stride,
            padding=1,
            transpose=transpose,
            act_norm=act_norm
        )

    def forward(self, x):
        y = self.conv(x)
        return y

class MultiGroupConv2d(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride,
        padding,
        groups,
        act_norm=False
    ):
        super(MultiGroupConv2d, self).__init__()
        self.act_norm = act_norm
        if in_channels % groups != 0:
            groups = 1
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=groups
        )
        self.norm = nn.GroupNorm(groups, out_channels)
        self.activate = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        y = self.conv(x)
        if self.act_norm:
            y = self.activate(self.norm(y))
        return y


class TurbulenceEncoder(nn.Module):
    def __init__(self, C_in, spatial_hidden_dim, num_spatial_layers):
        super(TurbulenceEncoder, self).__init__()
        strides = stride_generator(num_spatial_layers)
        self.enc = nn.Sequential(
            ConvDynamicsLayer(C_in, spatial_hidden_dim, stride=strides[0]),
            *[ConvDynamicsLayer(spatial_hidden_dim, spatial_hidden_dim, stride=s) for s in strides[1:]]
        )

    def forward(self, x):
        enc1 = self.enc[0](x)
        latent = enc1
        for i in range(1, len(self.enc)):
            latent = self.enc[i](latent)
        return latent, enc1

class TurbulenceDecoder(nn.Module):
    def __init__(self, spatial_hidden_dim, C_out, num_spatial_layers):
        super(TurbulenceDecoder, self).__init__()
        strides = stride_generator(num_spatial_layers, reverse=True)
        self.dec = nn.Sequential(
            *[ConvDynamicsLayer(spatial_hidden_dim, spatial_hidden_dim, stride=s, transpose=True) for s in strides[:-1]],
            ConvDynamicsLayer(2 * spatial_hidden_dim, spatial_hidden_dim, stride=strides[-1], transpose=True)
        )
        self.readout = nn.Conv2d(spatial_hidden_dim, C_out, 1)

    def forward(self, hid, enc1=None):
        for i in range(0, len(self.dec) - 1):
            hid = self.dec[i](hid)
        Y = self.dec[-1](torch.cat([hid, enc1], dim=1))
        Y = self.readout(Y)
        return Y

class Triton_Turbulence(nn.Module):
    def __init__(
        self,
        input_channel,
        spatial_hidden_dim=256,
        output_channels=4,
        temporal_hidden_dim=512,
        num_spatial_layers=4,
        num_temporal_layers=8,
        in_time_seq_length=10,
        out_time_seq_length=10
    ):
        super(Triton_Turbulence, self).__init__()
        T = 1
        C = input_channel
        self.output_dim = output_channels
        self.input_time_seq_length = in_time_seq_length
        self.output_time_seq_length = out_time_seq_length
        
        self.Turbulence_encoder = TurbulenceEncoder(C, spatial_hidden_dim, num_spatial_layers)
        self.LDC = Latent_Dynamical_Core(
            T * spatial_hidden_dim,
            temporal_hidden_dim,
            num_temporal_layers,
            input_resolution=None,
            mlp_ratio=4.0,
            drop_path=0.1
        )
        self.Turbulence_decoder = TurbulenceDecoder(spatial_hidden_dim, self.output_dim, num_spatial_layers)

    def forward(self, input_state):
        batch_size, temporal_length, channels, height, width = input_state.shape
        reshaped_input = input_state.view(batch_size * temporal_length, channels, height, width)
        
        encoded_features, skip_connection = self.Turbulence_encoder(reshaped_input)
        _, encoded_channels, encoded_height, encoded_width = encoded_features.shape
        encoded_features = encoded_features.view(batch_size, temporal_length, encoded_channels, encoded_height, encoded_width)
        
        temporal_bias = encoded_features
        temporal_hidden = self.LDC(temporal_bias)
        reshaped_hidden = temporal_hidden.view(batch_size * temporal_length, encoded_channels, encoded_height, encoded_width)

        decoded_output = self.Turbulence_decoder(reshaped_hidden, skip_connection)
        final_output = decoded_output.view(batch_size, temporal_length, -1, height, width)
        
        return final_output


import torch

def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params


if __name__ == '__main__':
    inputs = torch.randn(1, 1, 1, 128, 128).cuda()
    model = Triton_Turbulence(
        input_channel=1,
        spatial_hidden_dim=256,
        output_channels=1,
        temporal_hidden_dim=768,
        num_spatial_layers=4,
        num_temporal_layers=8).cuda()
    output = model(inputs)
    
    # Get total parameters
    total_params = count_parameters(model)
    
    # Calculate in millions and billions
    params_in_millions = total_params / 1_000_000
    params_in_billions = total_params / 1_000_000_000
    
    print("Output shape:", output.shape)
    print(f"Trainable parameters: {total_params:,}")
    print(f"Trainable parameters: {params_in_millions:.2f}M")
    print(f"Trainable parameters: {params_in_billions:.3f}B")

Output shape: torch.Size([1, 1, 1, 128, 128])
Trainable parameters: 55,593,985
Trainable parameters: 55.59M
Trainable parameters: 0.056B
